# Evaluación final del checkpoint MobileNetV2

Este cuaderno evalúa **`mejor_modelo_hojas_finetuning.keras`** sin entrenarlo ni modificar sus pesos. Calcula accuracy, precision, recall, F1 y matrices de confusión para los umbrales **0,50** (clasificación) y **0,90** (autorización de fertirriego en la simulación).

**Antes de ejecutarlo:** prepara en Google Drive una carpeta **nueva de prueba** con imágenes que nunca estuvieron en el entrenamiento ni validación:

```text
Mi unidad/
  IA_prueba_independiente/
    Sanas/
      imagen_sana_01.jpg
      ...
    Nutrientes/
      imagen_deficiencia_01.jpg
      ...
    manifest_prueba.csv         [opcional al ejecutar, necesario para trazabilidad completa]
```

**No copies directamente las 2.247 imágenes de `IA/Sanas` y `IA/Nutrientes`:** fueron utilizadas para construir el entrenamiento original y no tenemos constancia de qué fotos integraron sus conjuntos. Si no hay imágenes nuevas verificadas, solo es posible hacer una evaluación exploratoria, no una validación independiente.

El script compara huellas SHA-256 de prueba y entrenamiento cuando está disponible el inventario; esto **solo detecta copias idénticas**, no fotografías muy parecidas ni la misma hoja/planta. Debes conservar la procedencia de las imágenes y la etiqueta original para documentar su independencia.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Revisa estas rutas antes de ejecutar:
from pathlib import Path

CARPETA_PRUEBA = Path("/content/drive/MyDrive/IA_prueba_independiente")
MODELO_DRIVE = Path("/content/drive/MyDrive/IA/mejor_modelo_hojas_finetuning.keras")
INVENTARIO_ENTRENAMIENTO = Path("/content/drive/MyDrive/IA_documentacion/inventario_imagenes.csv")
SALIDA = Path("/content/drive/MyDrive/IA_evaluacion_final")

# Pon True SOLO si verificaste que todas las fotos de CARPETA_PRUEBA son nuevas
# y no se utilizaron ni en entrenamiento ni validacion, ni son variantes de
# los mismos originales o tomas de una misma planta ya utilizada.
CONFIRMO_IMAGENES_NO_USADAS = False

print("Imagenes para prueba:", CARPETA_PRUEBA)
print("Modelo en Drive:", MODELO_DRIVE)
print("Inventario original:", INVENTARIO_ENTRENAMIENTO)
print("Salida:", SALIDA)


In [ ]:
# Descargar solo el script de evaluacion y, si no esta en Drive, el modelo .keras
from urllib.request import urlretrieve

BASE = "https://raw.githubusercontent.com/pealvaga/riego-inteligente-guanabana-simulacion/main/05_MobileNetV2/"
SCRIPT = Path("/content/evaluar_checkpoint_final.py")
urlretrieve(BASE + "evaluar_checkpoint_final.py", SCRIPT)

if MODELO_DRIVE.is_file():
    MODELO = MODELO_DRIVE
else:
    MODELO = Path("/content/mejor_modelo_hojas_finetuning.keras")
    print("Modelo no encontrado en Drive: se descarga el publicado en GitHub.")
    urlretrieve(BASE + "mejor_modelo_hojas_finetuning.keras", MODELO)

print("Script:", SCRIPT)
print("Modelo:", MODELO, "| bytes:", MODELO.stat().st_size)


In [ ]:
# Ejecuta la evaluacion (sin reentrenar y sin modificar el modelo)
import subprocess
import sys

cmd = [
    sys.executable, str(SCRIPT),
    "--prueba", str(CARPETA_PRUEBA),
    "--modelo", str(MODELO),
    "--salida", str(SALIDA),
    "--inventario-entrenamiento", str(INVENTARIO_ENTRENAMIENTO),
]
if CONFIRMO_IMAGENES_NO_USADAS:
    cmd.append("--confirma-no-usadas")
subprocess.run(cmd, check=True)


In [ ]:
# Ver resultados de ambos umbrales
import json
import pandas as pd
from IPython.display import display, Image

resultados = json.loads((SALIDA / "metricas_mobilenetv2.json").read_text(encoding="utf-8"))
print("ESTADO:", resultados["estado"])
print("Total evaluado:", resultados["n_prueba"])
print("Sanas:", resultados["distribucion"]["sana"],
      "| Deficiencia:", resultados["distribucion"]["deficiencia"])
display(pd.read_csv(SALIDA / "metricas_resumen.csv"))
display(Image(filename=str(SALIDA / "matriz_confusion_05.png")))
display(Image(filename=str(SALIDA / "matriz_confusion_09.png")))


In [ ]:
# Descarga un ZIP ligero con CSV, JSON y graficos (SIN fotografias)
from google.colab import files
files.download(str(SALIDA / "evaluacion_mobilenetv2_checkpoint.zip"))


## Qué publicar en GitHub

Una vez revisada la independencia y las etiquetas, publica en **`07_Resultados/MobileNetV2/`**:
- `metricas_mobilenetv2.json` y `metricas_resumen.csv`
- `predicciones_mobilenetv2.csv` (rutas relativas, probabilidades, etiquetas)
- `matriz_confusion_05.png` y `matriz_confusion_09.png`
- `duplicados_en_prueba.csv` y `solapes_con_entrenamiento.csv`
- `resultados_por_fuente.csv` y el manifiesto de procedencia verificado

**No publiques las fotografías en GitHub si sus licencias no lo permiten.** Un modelo que arroja métricas altas sobre fotografías ya utilizadas para entrenar **no queda validado**. Si hay diferencias entre CoLeaf (café) y SoursopBD (guanábana), no interpretar automáticamente `Deficiencia` como diagnóstico nutricional específico del cultivo.
